# Assignment 8 — English → Gujarati Neural Machine Translation with OpenNMT-py
**ILLM** · **U23AI051**

* **Data:** `ilci` subset of [BPCC](https://huggingface.co/datasets/ai4bharat/BPCC) (`ilci/guj_Gujr.tsv`),
  English (`eng_Latn`) source → Gujarati (`guj_Gujr`, my native language) target.
* **Splits:** built **per length bin**, so train and test both contain sentences of every length.
  Length = number of whitespace-separated words in the English source sentence.

| bin | sentence length |
|---|---|
| bin1 | ≤ 10 |
| bin2 | (10, 20] |
| bin3 | (20, 30) |
| bin4 | ≥ 30 |

  (The assignment writes bin2 as `(10, 20)`, which leaves length 20 out of every bin; it is put in bin2.)
* **Model:** [OpenNMT-py](https://github.com/OpenNMT/OpenNMT-py) Transformer on SentencePiece BPE subwords.
* **Evaluation:** [sacreBLEU](https://github.com/mjpost/sacrebleu) on the full test set and on each bin.

## 0. Settings and imports

In [ ]:
# !pip install OpenNMT-py==3.5.1 sacrebleu sentencepiece huggingface_hub   (torch 2.2, Python 3.11)
import os, sys, csv, json, subprocess

import pandas as pd
import sentencepiece as spm
import sacrebleu
import torch
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

SEED = 42
TEST_FRAC, VALID_FRAC = 0.10, 0.01      # per bin; valid is taken from the training part
VOCAB = 8000                            # SentencePiece BPE vocabulary per language
TRAIN_STEPS = 10000

DATA = "data"
SPLIT = os.path.join(DATA, "splits")
RUN = "run"
os.makedirs(SPLIT, exist_ok=True)
os.makedirs(RUN, exist_ok=True)
ONMT = os.path.dirname(sys.executable)  # onmt_* scripts are installed next to python
BINS = ["bin1", "bin2", "bin3", "bin4"]

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Load the ILCI English–Gujarati data
BPCC is gated on Hugging Face: accept its terms once and set `HF_TOKEN` (or run `hf auth login`).

**Data-quality check — misaligned blocks.** A first model, trained with the same settings on *all*
107,385 pairs, was used to translate the whole corpus (greedy). Sentence-level chrF against the
reference, averaged over a 200-row window, is 40–80 almost everywhere but drops to **13–19** in four
contiguous blocks of the file. In those blocks the Gujarati sentence is a neighbouring or unrelated
sentence, not a translation of the English (even the *training* rows there score ≈ 14 chrF — the model
could not learn them). The blocks, padded by 100 rows, are removed below (≈ 18.5 % of the corpus).

| run | bin1 | bin2 | bin3 | bin4 | all |
|---|---|---|---|---|---|
| first model, unfiltered data (test BLEU) | 16.95 | 14.99 | 12.33 | 11.04 | 13.72 |

In [ ]:
path = hf_hub_download("ai4bharat/BPCC", "ilci/guj_Gujr.tsv", repo_type="dataset", local_dir=DATA)
df = pd.read_csv(path, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
print("raw pairs:", len(df))

df["src"] = df["src"].str.split().str.join(" ")    # collapse whitespace / newlines
df["tgt"] = df["tgt"].str.split().str.join(" ")

# file-row blocks whose Gujarati side is not a translation of the English (see the note above)
MISALIGNED = [(373, 1453), (15182, 16065), (86608, 95483), (98350, 107385)]
bad = df.index.to_series().apply(lambda i: any(a <= i < b for a, b in MISALIGNED))
print("misaligned pairs removed:", bad.sum())
print(df[bad][["src", "tgt"]].iloc[[0, -1]].to_string())

df = df[~bad & (df.src != "") & (df.tgt != "")].drop_duplicates(["src", "tgt"]).reset_index(drop=True)
print("clean pairs:", len(df))
df[["src", "tgt"]].head()

## 2. Length bins and train / valid / test splits

In [ ]:
def to_bin(n):
    if n <= 10:
        return "bin1"
    if n <= 20:
        return "bin2"
    if n < 30:
        return "bin3"
    return "bin4"


df["len"] = df["src"].str.split().str.len()
df["bin"] = df["len"].map(to_bin)

# split every bin separately -> each split has the same mix of lengths
parts = {"train": [], "valid": [], "test": []}
for b in BINS:
    g = df[df.bin == b].sample(frac=1, random_state=SEED)
    n_test, n_valid = round(len(g) * TEST_FRAC), round(len(g) * VALID_FRAC)
    parts["test"].append(g.iloc[:n_test])
    parts["valid"].append(g.iloc[n_test:n_test + n_valid])
    parts["train"].append(g.iloc[n_test + n_valid:])
splits = {k: pd.concat(v).reset_index(drop=True) for k, v in parts.items()}

for name, s in splits.items():
    for lang, col in [("en", "src"), ("gu", "tgt")]:
        with open(os.path.join(SPLIT, f"{name}.{lang}"), "w", encoding="utf-8") as f:
            f.write("\n".join(s[col]) + "\n")
splits["test"]["bin"].to_csv(os.path.join(SPLIT, "test.bin"), index=False, header=False)

counts = pd.DataFrame({k: s["bin"].value_counts() for k, s in splits.items()}).loc[BINS]
counts.loc["total"] = counts.sum()
counts["avg len (test)"] = list(splits["test"].groupby("bin")["len"].mean().round(1)) + [
    round(splits["test"]["len"].mean(), 1)]
counts

## 3. Subword segmentation (SentencePiece BPE)
One BPE model per language, trained on the training split only.

In [ ]:
sp = {}
for lang in ["en", "gu"]:
    spm.SentencePieceTrainer.train(
        input=os.path.join(SPLIT, f"train.{lang}"), model_prefix=os.path.join(RUN, f"spm.{lang}"),
        vocab_size=VOCAB, model_type="bpe", character_coverage=1.0, input_sentence_size=10**6,
        shuffle_input_sentence=True, minloglevel=2)
    sp[lang] = spm.SentencePieceProcessor(model_file=os.path.join(RUN, f"spm.{lang}.model"))
    for name in ["train", "valid", "test"]:
        lines = open(os.path.join(SPLIT, f"{name}.{lang}"), encoding="utf-8").read().splitlines()
        with open(os.path.join(SPLIT, f"{name}.sp.{lang}"), "w", encoding="utf-8") as f:
            f.write("\n".join(" ".join(p) for p in sp[lang].encode(lines, out_type=str)) + "\n")

ex = splits["train"].iloc[0]
print(ex.src, "->", " ".join(sp["en"].encode(ex.src, out_type=str)))
print(ex.tgt, "->", " ".join(sp["gu"].encode(ex.tgt, out_type=str)))

## 4. OpenNMT-py Transformer

6-layer encoder / 6-layer decoder Transformer (d_model 256, 4 heads, FFN 1024), sized to train on a
6 GB laptop GPU. Noam schedule, label smoothing 0.1, fp16, token-based batches.

In [ ]:
config = f"""
save_data: {RUN}/vocab
src_vocab: {RUN}/vocab.src
tgt_vocab: {RUN}/vocab.tgt
src_vocab_size: {VOCAB + 100}
tgt_vocab_size: {VOCAB + 100}
overwrite: true

data:
  corpus_1:
    path_src: {SPLIT}/train.sp.en
    path_tgt: {SPLIT}/train.sp.gu
    transforms: [filtertoolong]
  valid:
    path_src: {SPLIT}/valid.sp.en
    path_tgt: {SPLIT}/valid.sp.gu
src_seq_length: 200
tgt_seq_length: 200

save_model: {RUN}/model
save_checkpoint_steps: 5000
keep_checkpoint: 2
train_steps: {TRAIN_STEPS}
valid_steps: 2000
report_every: 500
seed: {SEED}
world_size: 1
gpu_ranks: [0]
num_workers: 0

encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 4
hidden_size: 256
word_vec_size: 256
transformer_ff: 1024
dropout: [0.1]
attention_dropout: [0.1]

batch_type: tokens
batch_size: 4096
valid_batch_size: 2048
accum_count: [2]
normalization: tokens
model_dtype: fp16
optim: adam
adam_beta2: 0.998
decay_method: noam
learning_rate: 2.0
warmup_steps: 4000
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
"""
with open(os.path.join(RUN, "config.yaml"), "w") as f:
    f.write(config)


def onmt(cmd, *args, log=None):
    out = open(log, "w") if log else subprocess.DEVNULL
    subprocess.run([os.path.join(ONMT, cmd), *args], check=True, stdout=out, stderr=subprocess.STDOUT)


onmt("onmt_build_vocab", "-config", f"{RUN}/config.yaml", "-n_sample", "-1")
print("vocab sizes:", sum(1 for _ in open(f"{RUN}/vocab.src")), sum(1 for _ in open(f"{RUN}/vocab.tgt")))

In [ ]:
MODEL = f"{RUN}/model_step_{TRAIN_STEPS}.pt"
if not os.path.exists(MODEL):          # training takes ~70 min; skip if the checkpoint already exists
    onmt("onmt_train", "-config", f"{RUN}/config.yaml", log=f"{RUN}/train.log")

for line in open(f"{RUN}/train.log", encoding="utf-8"):
    if "Validation" in line or ("Step" in line and f"/{TRAIN_STEPS};" in line):
        print(line.split("INFO] ")[-1].rstrip())

## 5. Translate the test set (beam search, beam = 5)

In [ ]:
PRED = f"{RUN}/pred.sp.gu"
if not os.path.exists(PRED):
    onmt("onmt_translate", "-model", MODEL, "-src", f"{SPLIT}/test.sp.en", "-output", PRED + ".tmp",
         "-gpu", "0", "-beam_size", "5", "-batch_type", "tokens", "-batch_size", "4096",
         "-max_length", "250", log=f"{RUN}/translate.log")
    os.replace(PRED + ".tmp", PRED)     # only a finished translation counts as cached

test = splits["test"].copy()
test["hyp"] = [sp["gu"].decode(l.split()) for l in open(PRED, encoding="utf-8").read().splitlines()]
assert len(test) == len(open(f"{SPLIT}/test.en", encoding="utf-8").read().splitlines())

## 6. BLEU — overall and bin-wise
sacreBLEU with its default `13a` tokenisation on the SentencePiece-decoded output. The ILCI references
already have punctuation split off (`શું થયું ?`) and the model learns to reproduce that style, so
`force=True` silences sacreBLEU's "tokenised input" warning.

In [ ]:
rows = []
for name, g in [*((b, test[test.bin == b]) for b in BINS), ("all", test)]:
    bleu = sacrebleu.corpus_bleu(list(g.hyp), [list(g.tgt)], force=True)
    rows.append({"bin": name, "sentences": len(g), "avg src len": round(g.len.mean(), 1),
                 "BLEU": round(bleu.score, 2),
                 "1/2/3/4-gram precision": "/".join(f"{p:.1f}" for p in bleu.precisions),
                 "BP": round(bleu.bp, 3), "hyp/ref len": round(bleu.sys_len / bleu.ref_len, 3)})
results = pd.DataFrame(rows).set_index("bin")
print(sacrebleu.corpus_bleu(list(test.hyp), [list(test.tgt)], force=True).format(signature=True))
json.dump(rows, open("ass8_results.json", "w"), indent=2)
results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(BINS, results.loc[BINS, "BLEU"], color="#4C72B0")
ax.axhline(results.loc["all", "BLEU"], color="gray", ls="--", label=f"overall {results.loc['all', 'BLEU']}")
for i, v in enumerate(results.loc[BINS, "BLEU"]):
    ax.text(i, v + 0.3, f"{v}", ha="center")
ax.set_xticks(range(4), ["≤10", "11–20", "21–29", "≥30"])
ax.set_xlabel("English sentence length (words)")
ax.set_ylabel("BLEU")
ax.set_title("Bin-wise BLEU, English → Gujarati")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Sample translations from each bin

In [ ]:
pd.set_option("display.max_colwidth", 200)
test[["bin", "src", "tgt", "hyp"]].groupby("bin").sample(2, random_state=SEED)

## 8. Bin-wise result analysis

ANALYSIS_PLACEHOLDER